# 01 — EDA (o que alimenta o `docs/01_eda.md`)

Kernel: o `.venv` deste case (`pip install -r requirements.txt`).
Cada bloco mostra a conta; `docs/01_eda.md` só resume.

In [ ]:
import pandas as pd
from case_paths import case_root
from incrementality.clean import clean_produtos, clean_projetos, clean_similares, clean_vendas
from incrementality.io import load_raw

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 80)

case = case_root()
raw = load_raw(case / "data" / "raw")
print("case:", case)
print({k: v.shape for k, v in raw.items()})

## Desconto nulo: a conta (antes de preencher)

Se `praticado == tabela` quando o desconto está vazio, nulo é zero. Não é convenção.

In [ ]:
vnd = raw["tb_vendas"].copy()
tabela = pd.to_numeric(vnd["vlr_venda_tabela"], errors="coerce")
praticado = pd.to_numeric(vnd["vlr_venda_praticado"], errors="coerce")
desc = pd.to_numeric(vnd["vlr_venda_desconto"], errors="coerce")
nulo = vnd["vlr_venda_desconto"].isna()

delta_nulo = (praticado[nulo] - tabela[nulo]).abs()
delta_cheio = (praticado[desc > 0] - (tabela[desc > 0] - desc[desc > 0])).abs()

resumo_desc = pd.DataFrame(
    {
        "recorte": ["desconto nulo", "desconto > 0"],
        "n": [int(nulo.sum()), int((desc > 0).sum())],
        "conta": ["praticado - tabela", "praticado - (tabela - desconto)"],
        "max_erro": [float(delta_nulo.max()), float(delta_cheio.max())],
        "n_erro_gt_0_01": [int((delta_nulo > 0.01).sum()), int((delta_cheio > 0.01).sum())],
    }
)
resumo_desc

## Ciclo: texto bruto no xlsx (`202401` e `2024/01`)

Mesmo arquivo que `data/csv/ciclos_tb_vendas.csv`.

In [ ]:
ciclos = (
    vnd["cod_ciclo"].astype(str).str.strip()
    .value_counts()
    .rename_axis("cod_ciclo_bruto")
    .reset_index(name="n")
    .sort_values("cod_ciclo_bruto")
)
ciclos["tem_barra"] = ciclos["cod_ciclo_bruto"].str.contains("/")
display(ciclos.head(20))
print("valores distintos brutos:", len(ciclos), "| com barra:", int(ciclos.tem_barra.sum()), "linhas com barra:", int(ciclos.loc[ciclos.tem_barra, "n"].sum()))

## Cadastro: vírgula no preço e barra no phase-in

A primeira limpeza tinha chamado isso de "preço ausente" / SKU sem data.

In [ ]:
prod = raw["tb_produtos_atributos"]
print("preço com vírgula")
display(prod.loc[prod["preco_regular"].astype(str).str.contains(","), ["cod_sku", "nome_produto", "preco_regular"]])
print("phase_in com barra")
display(prod.loc[prod["cod_ciclo_phase_in"].astype(str).str.contains("/"), ["cod_sku", "nome_produto", "subcategoria", "cod_ciclo_phase_in"]])

## Depois da limpeza (parsers)

Incumbente / projeto / outra inovação — números que o `01_eda.md` usa.

In [ ]:
projetos = clean_projetos(raw["tb_projetos_lancamento"])
produtos = clean_produtos(raw["tb_produtos_atributos"], set(projetos["cod_sku_lancamento"]))
vendas = clean_vendas(raw["tb_vendas"])
similares = clean_similares(raw["tb_skus_similares"])

print("ciclo nulo após mapa:", int(vendas.flag_ciclo_nulo.sum()), "| ciclos distintos:", vendas.cod_ciclo.nunique())
print("preço NaN:", int(produtos.preco_regular.isna().sum()), "| phase_in NaN:", int(produtos.cod_ciclo_phase_in.isna().sum()))
comp = produtos[["is_project", "is_incumbent", "is_other_innovation"]].sum().rename("n").to_frame()
comp

## Marca, canal, similares

In [ ]:
print("marca bruta")
display(raw["tb_produtos_atributos"]["marca"].value_counts())
print("canal bruto")
display(raw["tb_vendas"]["canal_venda"].value_counts())
print("similares: score nulo", similares.score_similaridade_par.isna().sum(), "de", len(similares))
print("projetos sem lista como principal",
      (~projetos.cod_sku_lancamento.isin(similares.cod_sku_principal)).sum(), "de", len(projetos))